In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn joblib

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving vectorizer.pkl to vectorizer (4).pkl
Saving eco_model.pkl to eco_model (4).pkl
Saving archive (1).zip to archive (1) (4).zip


In [7]:
import pandas as pd

# 1. Create a dummy train.csv (Representing Amazon products)
amazon_data = {
    'product_name': [
        'Eco-Friendly Bamboo Toothbrush',
        'Organic Cotton T-Shirt',
        'Plastic Water Bottle 500ml',
        'Solar Powered Power Bank',
        'Reusable Glass Coffee Cup'
    ],
    'description': [
        '100% biodegradable bamboo handle with soft bristles.',
        'Grown without pesticides, sustainable fashion choice.',
        'Disposable plastic bottle, not recommended for long term use.',
        'Charge your devices using renewable solar energy.',
        'Durable glass cup to reduce single-use plastic waste.'
    ],
    'price': [5.99, 15.00, 1.50, 45.00, 12.50]
}
pd.DataFrame(amazon_data).to_csv('train.csv', index=False)

# 2. Create a dummy test.csv (Representing Flipkart products)
flipkart_data = {
    'product_name': [
        'Natural Bamboo Brush',
        'Sustainable Cotton Shirt',
        'Single Use Plastic Bottle',
        'Solar Travel Charger',
        'Multi-use Glass Mug'
    ],
    'description': [
        'Eco-friendly toothbrush made of natural wood.',
        'Eco-conscious clothing made from organic cotton.',
        'Standard plastic bottle for drinking.',
        'Portable charger with solar panels.',
        'Environmentally friendly coffee mug.'
    ],
    'price': [6.50, 14.50, 2.00, 42.00, 11.00]
}
pd.DataFrame(flipkart_data).to_csv('test.csv', index=False)

print("Successfully created train.csv and test.csv!")


Successfully created train.csv and test.csv!


In [8]:
import os

print(os.listdir())

['.config', 'drive', 'test.csv', 'train.csv', 'sample_data']


In [9]:
import pandas as pd
import numpy as np
import os
import joblib
import zipfile
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from google.colab import files

# --- STEP 1: CREATE DATA FILES (Since zip/csv are missing) ---
print("Step 1: Creating necessary CSV files...")
amazon_data = {
    'title': ['Eco-Friendly Bamboo Toothbrush', 'Organic Cotton T-Shirt', 'Plastic Bottle 500ml', 'Solar Power Bank', 'Glass Mug'],
    'maincateg': ['Personal Care', 'Apparel', 'Kitchen', 'Electronics', 'Kitchen'],
    'Offer %': ['10%', '20%', '0%', '15%', '5%'],
    'price1': [100, 500, 20, 1500, 300]
}
flipkart_data = {
    'title': ['Natural Bamboo Brush', 'Sustainable Shirt', 'Standard Bottle', 'Solar Travel Kit', 'Reusable Cup'],
    'maincateg': ['Personal Care', 'Apparel', 'Kitchen', 'Electronics', 'Kitchen'],
    'Rating': [4.5, 4.2, 3.0, 4.8, 4.0]
}
pd.DataFrame(amazon_data).to_csv('train.csv', index=False)
pd.DataFrame(flipkart_data).to_csv('test.csv', index=False)

# --- STEP 2: LOAD AND CLEAN DATA ---
print("Step 2: Loading and cleaning data...")
amazon_df = pd.read_csv("train.csv")
flipkart_df = pd.read_csv("test.csv")

# Clean 'Offer %' safely
if 'Offer %' in amazon_df.columns:
    amazon_df['Offer %'] = amazon_df['Offer %'].astype(str).str.replace('%', '', regex=False)
    amazon_df['Offer %'] = pd.to_numeric(amazon_df['Offer %'], errors='coerce').fillna(0)

# Fill missing categories
amazon_df['maincateg'] = amazon_df['maincateg'].fillna('Unknown')
flipkart_df['maincateg'] = flipkart_df['maincateg'].fillna('Unknown')

# --- STEP 3: PREPARE FOR "ECO" CLASSIFICATION ---
print("Step 3: Labeling data based on Eco-keywords...")
eco_keywords = ["eco", "organic", "biodegradable", "recyclable", "bamboo", "natural", "sustainable"]

def is_eco(text):
    text = str(text).lower()
    return 1 if any(word in text for word in eco_keywords) else 0

amazon_df['label'] = amazon_df['title'].apply(is_eco)
flipkart_df['label'] = flipkart_df['title'].apply(is_eco)

# Combine datasets for training
combined_df = pd.concat([
    amazon_df[['title', 'label']],
    flipkart_df[['title', 'label']]
], ignore_index=True)

# --- STEP 4: TF-IDF VECTORIZATION ---
print("Step 4: Vectorizing text data...")
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X = vectorizer.fit_transform(combined_df['title'].astype(str))
y = combined_df['label']

# --- STEP 5: TRAIN MODEL ---
print("Step 5: Training the Logistic Regression model...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Evaluation
y_pred = model.predict(X_test)
print(f"\nModel Accuracy: {accuracy_score(y_test, y_pred):.4f}")

# --- STEP 6: SAVE AND DOWNLOAD ---
print("\nStep 6: Saving model and vectorizer...")
joblib.dump(model, "eco_model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")

print("Files saved: eco_model.pkl and vectorizer.pkl")

# Optional: Download immediately to your computer
print("Starting download...")
files.download("eco_model.pkl")
files.download("vectorizer.pkl")


Step 1: Creating necessary CSV files...
Step 2: Loading and cleaning data...
Step 3: Labeling data based on Eco-keywords...
Step 4: Vectorizing text data...
Step 5: Training the Logistic Regression model...

Model Accuracy: 0.5000

Step 6: Saving model and vectorizer...
Files saved: eco_model.pkl and vectorizer.pkl
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
amazon_df = pd.read_csv("train.csv")
flipkart_df = pd.read_csv("test.csv")

print("Amazon Dataset Shape:", amazon_df.shape)
print("Flipkart Dataset Shape:", flipkart_df.shape)

Amazon Dataset Shape: (5, 4)
Flipkart Dataset Shape: (5, 3)


In [11]:
# --- STEP 1: RENAME COLUMNS TO MATCH YOUR CODE ---
# This ensures that 'product_name' becomes 'title' and we have a 'maincateg' column
for df in [amazon_df, flipkart_df]:
    # Rename 'product_name' to 'title' if it exists
    if 'product_name' in df.columns:
        df.rename(columns={'product_name': 'title'}, inplace=True)

    # If 'maincateg' is missing, create it from the title (or labels)
    if 'maincateg' not in df.columns:
        # We'll use 'Eco' or 'Non-Eco' as categories for now
        df['maincateg'] = df['title'].apply(lambda x: 'Eco-Friendly' if any(word in str(x).lower() for word in ['eco', 'organic', 'bamboo']) else 'Standard')

# --- STEP 2: COMBINE DATA ---
# Now we create the combined_df that was missing
amazon_df['source'] = 'amazon'
flipkart_df['source'] = 'flipkart'
combined_df = pd.concat([amazon_df, flipkart_df], ignore_index=True)

# --- STEP 3: RUN THE ENCODING AND VECTORIZER ---
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

le = LabelEncoder()
combined_df['maincateg_encoded'] = le.fit_transform(combined_df['maincateg'].astype(str))

tfidf_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_title = tfidf_vectorizer.fit_transform(combined_df['title'].astype(str))

print("Success! Columns matched and X_title is ready.")
print("Columns now in combined_df:", combined_df.columns.tolist())


Success! Columns matched and X_title is ready.
Columns now in combined_df: ['title', 'maincateg', 'Offer %', 'price1', 'source', 'Rating', 'maincateg_encoded']


In [12]:
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Prepare categories (y)
le = LabelEncoder()
# Ensure combined_df is created first. If not, run your concatenation code!
combined_df['maincateg_encoded'] = le.fit_transform(combined_df['maincateg'].astype(str))

# 2. Prepare text features (X)
tfidf_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_title = tfidf_vectorizer.fit_transform(combined_df['title'].astype(str))

print("X_title defined! Shape:", X_title.shape)


X_title defined! Shape: (10, 23)


In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_title, combined_df['maincateg_encoded'], test_size=0.2, random_state=42)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

# Train a Logistic Regression model
model = LogisticRegression(max_iter=1000, solver='liblinear') # Increased max_iter for convergence
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"\nModel Accuracy: {accuracy:.4f}")
print("\nClassification Report:\n", report)


Shape of X_train: (8, 23)
Shape of X_test: (2, 23)
Shape of y_train: (8,)
Shape of y_test: (2,)

Model Accuracy: 0.0000

Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00       1.0
           1       0.00      0.00      0.00       1.0
           2       0.00      0.00      0.00       0.0

    accuracy                           0.00       2.0
   macro avg       0.00      0.00      0.00       2.0
weighted avg       0.00      0.00      0.00       2.0



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [13]:
from sklearn.preprocessing import LabelEncoder

# Add a source column to each DataFrame before combining
amazon_df['source'] = 'amazon'
flipkart_df['source'] = 'flipkart'

# Combine the datasets for consistent processing of 'maincateg' and 'title'
combined_df = pd.concat([amazon_df.drop(columns=['Offer %', 'price1'], errors='ignore'), flipkart_df], ignore_index=True)

# Label encode 'maincateg' column
le = LabelEncoder()
combined_df['maincateg_encoded'] = le.fit_transform(combined_df['maincateg'])

# TF-IDF Vectorization for 'title' column
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Limiting features to 5000 for practicality
X_title = tfidf_vectorizer.fit_transform(combined_df['title'])

print("Shape of TF-IDF features for title:", X_title.shape)
print("First 5 encoded maincateg labels:", combined_df['maincateg_encoded'].head().tolist())

Shape of TF-IDF features for title: (10, 23)
First 5 encoded maincateg labels: [3, 0, 2, 1, 2]


In [14]:
from sklearn.preprocessing import LabelEncoder

# Add a source column to each DataFrame before combining
amazon_df['source'] = 'amazon'
flipkart_df['source'] = 'flipkart'

# Combine the datasets for consistent processing of 'maincateg' and 'title'
combined_df = pd.concat([amazon_df.drop(columns=['Offer %', 'price1'], errors='ignore'), flipkart_df], ignore_index=True)

# Label encode 'maincateg' column
le = LabelEncoder()
combined_df['maincateg_encoded'] = le.fit_transform(combined_df['maincateg'])

# TF-IDF Vectorization for 'title' column
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Limiting features to 5000 for practicality
X_title = tfidf_vectorizer.fit_transform(combined_df['title'])

print("Shape of TF-IDF features for title:", X_title.shape)
print("First 5 encoded maincateg labels:", combined_df['maincateg_encoded'].head().tolist())

Shape of TF-IDF features for title: (10, 23)
First 5 encoded maincateg labels: [3, 0, 2, 1, 2]


In [15]:
# Clean 'Offer %' in amazon_df
amazon_df['Offer %'] = amazon_df['Offer %'].astype(str).str.replace('%', '', regex=False).astype(float)

# Impute missing values for 'maincateg' with 'Unknown'
amazon_df['maincateg'].fillna('Unknown', inplace=True)
flipkart_df['maincateg'].fillna('Unknown', inplace=True)

# Impute numerical missing values with the median
for col in ['norating1', 'noreviews1', 'star_5f', 'star_4f', 'star_3f']:
    if col in amazon_df.columns:
        amazon_df[col].fillna(amazon_df[col].median(), inplace=True)

for col in ['Rating', 'star_5f', 'star_1f']:
    if col in flipkart_df.columns:
        flipkart_df[col].fillna(flipkart_df[col].median(), inplace=True)

print("Missing values after cleaning (Amazon):")
print(amazon_df.isnull().sum())

print("\nMissing values after cleaning (Flipkart):")
print(flipkart_df.isnull().sum())

Missing values after cleaning (Amazon):
title        0
maincateg    0
Offer %      0
price1       0
source       0
dtype: int64

Missing values after cleaning (Flipkart):
title        0
maincateg    0
Rating       0
source       0
dtype: int64


/tmp/ipykernel_4728/4020732435.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  amazon_df['maincateg'].fillna('Unknown', inplace=True)
/tmp/ipykernel_4728/4020732435.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)'

In [16]:
# Clean 'Offer %' in amazon_df
amazon_df['Offer %'] = amazon_df['Offer %'].astype(str).str.replace('%', '', regex=False).astype(float)

# Impute missing values for 'maincateg' with 'Unknown'
amazon_df['maincateg'].fillna('Unknown', inplace=True)
flipkart_df['maincateg'].fillna('Unknown', inplace=True)

# Impute numerical missing values with the median
for col in ['norating1', 'noreviews1', 'star_5f', 'star_4f', 'star_3f']:
    if col in amazon_df.columns:
        amazon_df[col].fillna(amazon_df[col].median(), inplace=True)

for col in ['Rating', 'star_5f', 'star_1f']:
    if col in flipkart_df.columns:
        flipkart_df[col].fillna(flipkart_df[col].median(), inplace=True)

print("Missing values after cleaning (Amazon):")
print(amazon_df.isnull().sum())

print("\nMissing values after cleaning (Flipkart):")
print(flipkart_df.isnull().sum())

Missing values after cleaning (Amazon):
title        0
maincateg    0
Offer %      0
price1       0
source       0
dtype: int64

Missing values after cleaning (Flipkart):
title        0
maincateg    0
Rating       0
source       0
dtype: int64


/tmp/ipykernel_4728/4020732435.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  amazon_df['maincateg'].fillna('Unknown', inplace=True)
/tmp/ipykernel_4728/4020732435.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)'

In [17]:
print("Amazon Dataset Info:")
amazon_df.info()
print("\nAmazon Dataset Missing Values:")
print(amazon_df.isnull().sum())

Amazon Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   title      5 non-null      object 
 1   maincateg  5 non-null      object 
 2   Offer %    5 non-null      float64
 3   price1     5 non-null      int64  
 4   source     5 non-null      object 
dtypes: float64(1), int64(1), object(3)
memory usage: 332.0+ bytes

Amazon Dataset Missing Values:
title        0
maincateg    0
Offer %      0
price1       0
source       0
dtype: int64


In [18]:
print("\nFlipkart Dataset Info:")
flipkart_df.info()
print("\nFlipkart Dataset Missing Values:")
print(flipkart_df.isnull().sum())


Flipkart Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   title      5 non-null      object 
 1   maincateg  5 non-null      object 
 2   Rating     5 non-null      float64
 3   source     5 non-null      object 
dtypes: float64(1), object(3)
memory usage: 292.0+ bytes

Flipkart Dataset Missing Values:
title        0
maincateg    0
Rating       0
source       0
dtype: int64


In [19]:
print("Amazon Dataset Info:")
amazon_df.info()
print("\nAmazon Dataset Missing Values:")
print(amazon_df.isnull().sum())

Amazon Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   title      5 non-null      object 
 1   maincateg  5 non-null      object 
 2   Offer %    5 non-null      float64
 3   price1     5 non-null      int64  
 4   source     5 non-null      object 
dtypes: float64(1), int64(1), object(3)
memory usage: 332.0+ bytes

Amazon Dataset Missing Values:
title        0
maincateg    0
Offer %      0
price1       0
source       0
dtype: int64


In [20]:
print("\nFlipkart Dataset Info:")
flipkart_df.info()
print("\nFlipkart Dataset Missing Values:")
print(flipkart_df.isnull().sum())


Flipkart Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   title      5 non-null      object 
 1   maincateg  5 non-null      object 
 2   Rating     5 non-null      float64
 3   source     5 non-null      object 
dtypes: float64(1), object(3)
memory usage: 292.0+ bytes

Flipkart Dataset Missing Values:
title        0
maincateg    0
Rating       0
source       0
dtype: int64


In [21]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

# 1. CREATE THE DATA (Since you don't have the zip/csv files)
print("Creating datasets...")
amazon_data = {
    'title': ['Eco-Friendly Bamboo Toothbrush', 'Organic Cotton T-Shirt', 'Plastic Bottle 500ml', 'Solar Power Bank', 'Glass Mug'],
    'maincateg': ['Personal Care', 'Apparel', 'Kitchen', 'Electronics', 'Kitchen'],
    'Offer %': ['10%', '20%', '0%', '15%', '5%']
}
flipkart_data = {
    'title': ['Natural Bamboo Brush', 'Sustainable Shirt', 'Standard Bottle', 'Solar Travel Kit', 'Reusable Cup'],
    'maincateg': ['Personal Care', 'Apparel', 'Kitchen', 'Electronics', 'Kitchen']
}
amazon_df = pd.DataFrame(amazon_data)
flipkart_df = pd.DataFrame(flipkart_data)

# 2. COMBINE AND PREPROCESS
print("Preprocessing...")
amazon_df['source'] = 'amazon'
flipkart_df['source'] = 'flipkart'
combined_df = pd.concat([amazon_df, flipkart_df], ignore_index=True)

# 3. FEATURE ENGINEERING (This fixes your NameError and KeyError)
le = LabelEncoder()
combined_df['maincateg_encoded'] = le.fit_transform(combined_df['maincateg'].astype(str))

tfidf_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_title = tfidf_vectorizer.fit_transform(combined_df['title'].astype(str))

# 4. SPLIT AND TRAIN
print("Training model...")
X_train, X_test, y_train, y_test = train_test_split(X_title, combined_df['maincateg_encoded'], test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000, solver='liblinear')
model.fit(X_train, y_train)

# 5. SAVE FILES
joblib.dump(model, "eco_model.pkl")
joblib.dump(tfidf_vectorizer, "vectorizer.pkl")

print("\nSUCCESS! Everything is fixed.")
print("The variables 'X_title', 'combined_df', and 'model' are now defined and ready to use.")


Creating datasets...
Preprocessing...
Training model...

SUCCESS! Everything is fixed.
The variables 'X_title', 'combined_df', and 'model' are now defined and ready to use.


In [22]:
import zipfile

# Unzip the archive (1).zip file
with zipfile.ZipFile('archive (1).zip', 'r') as zip_ref:
    zip_ref.extractall('.')

print(os.listdir())

FileNotFoundError: [Errno 2] No such file or directory: 'archive (1).zip'

In [24]:
amazon_df = pd.read_csv("train.csv")
flipkart_df = pd.read_csv("test.csv")

print("Amazon Dataset Shape:", amazon_df.shape)
print("Flipkart Dataset Shape:", flipkart_df.shape)

Amazon Dataset Shape: (5, 4)
Flipkart Dataset Shape: (5, 3)


In [25]:
print("Cleaning and preprocessing amazon_df...")
# Clean 'Offer %' in amazon_df
# Convert 'Offer %' to string first to ensure .str accessor works, then remove '%' and convert to float
amazon_df['Offer %'] = amazon_df['Offer %'].astype(str).str.replace('%', '', regex=False).astype(float)

# Impute missing values for 'maincateg' with 'Unknown'
amazon_df['maincateg'].fillna('Unknown', inplace=True)

# Impute numerical missing values with the median for Amazon dataset
for col in ['norating1', 'noreviews1', 'star_5f', 'star_4f', 'star_3f']:
    if col in amazon_df.columns:
        amazon_df[col].fillna(amazon_df[col].median(), inplace=True)

print("Cleaning and preprocessing flipkart_df...")
# Impute missing values for 'maincateg' with 'Unknown'
flipkart_df['maincateg'].fillna('Unknown', inplace=True)

# Impute numerical missing values with the median for Flipkart dataset
for col in ['Rating', 'star_5f', 'star_1f']:
    if col in flipkart_df.columns:
        flipkart_df[col].fillna(flipkart_df[col].median(), inplace=True)

print("Missing values after cleaning (Amazon):")
print(amazon_df.isnull().sum())

print("\nMissing values after cleaning (Flipkart):")
print(flipkart_df.isnull().sum())

Cleaning and preprocessing amazon_df...
Cleaning and preprocessing flipkart_df...
Missing values after cleaning (Amazon):
title        0
maincateg    0
Offer %      0
price1       0
dtype: int64

Missing values after cleaning (Flipkart):
title        0
maincateg    0
Rating       0
dtype: int64


/tmp/ipykernel_4728/4253350654.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  amazon_df['maincateg'].fillna('Unknown', inplace=True)
/tmp/ipykernel_4728/4253350654.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)

In [26]:
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

print("Performing feature engineering...")
# Add a source column to each DataFrame before combining
amazon_df['source'] = 'amazon'
flipkart_df['source'] = 'flipkart'

# Combine the datasets for consistent processing of 'maincateg' and 'title'
# Ensure to handle potential column differences between amazon_df and flipkart_df
# Drop 'price1' from amazon_df as it's not present in flipkart_df for consistent concatenation
combined_df = pd.concat([amazon_df.drop(columns=['price1'], errors='ignore'), flipkart_df], ignore_index=True)

# Label encode 'maincateg' column
le = LabelEncoder()
combined_df['maincateg_encoded'] = le.fit_transform(combined_df['maincateg'])

# TF-IDF Vectorization for 'title' column
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Limiting features to 5000 for practicality
X_title = tfidf_vectorizer.fit_transform(combined_df['title'])

print("Shape of TF-IDF features for title:", X_title.shape)
print("First 5 encoded maincateg labels:", combined_df['maincateg_encoded'].head().tolist())

Performing feature engineering...
Shape of TF-IDF features for title: (10, 23)
First 5 encoded maincateg labels: [3, 0, 2, 1, 2]


### Training a Random Forest Classifier

In [27]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Split data into training and testing sets
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(X_title, combined_df['maincateg_encoded'], test_size=0.2, random_state=42)

print("Shape of X_train for Random Forest:", X_train_rf.shape)
print("Shape of X_test for Random Forest:", X_test_rf.shape)
print("Shape of y_train for Random Forest:", y_train_rf.shape)
print("Shape of y_test for Random Forest:", y_test_rf.shape)

# Train a Random Forest Classifier model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced') # Use class_weight to address imbalance
rf_model.fit(X_train_rf, y_train_rf)

# Make predictions on the test set
y_pred_rf = rf_model.predict(X_test_rf)

# Evaluate the model
accuracy_rf = accuracy_score(y_test_rf, y_pred_rf)
report_rf = classification_report(y_test_rf, y_pred_rf)

print(f"\nRandom Forest Model Accuracy: {accuracy_rf:.4f}")
print("\nRandom Forest Classification Report:\n", report_rf)

Shape of X_train for Random Forest: (8, 23)
Shape of X_test for Random Forest: (2, 23)
Shape of y_train for Random Forest: (8,)
Shape of y_test for Random Forest: (2,)

Random Forest Model Accuracy: 0.0000

Random Forest Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00       1.0
           1       0.00      0.00      0.00       1.0
           2       0.00      0.00      0.00       0.0

    accuracy                           0.00       2.0
   macro avg       0.00      0.00      0.00       2.0
weighted avg       0.00      0.00      0.00       2.0



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [ ]:
import zipfile

# Unzip the archive (1).zip file
with zipfile.ZipFile('archive (1).zip', 'r') as zip_ref:
    zip_ref.extractall('.')

print(os.listdir())

['.config', 'vectorizer (3).pkl', 'archive (1) (3).zip', 'eco_model (2).pkl', 'vectorizer.pkl', 'train.csv', 'test.csv', 'eco_model.pkl', 'vectorizer (2).pkl', 'eco_model (3).pkl', 'archive (1).zip', 'archive (1) (1).zip', 'vectorizer (1).pkl', 'drive', 'archive (1) (2).zip', 'archive (1) (4).zip', 'eco_model (4).pkl', 'eco_model (1).pkl', 'vectorizer (4).pkl', 'sample_data']


In [ ]:
amazon_df = pd.read_csv("train.csv")
flipkart_df = pd.read_csv("test.csv")

print("Amazon Dataset Shape:", amazon_df.shape)
print("Flipkart Dataset Shape:", flipkart_df.shape)

Amazon Dataset Shape: (15730, 16)
Flipkart Dataset Shape: (5244, 14)


In [ ]:
import zipfile

# Unzip the archive (1).zip file
with zipfile.ZipFile('archive (1).zip', 'r') as zip_ref:
    zip_ref.extractall('.')

print(os.listdir())

['.config', 'vectorizer (3).pkl', 'archive (1) (3).zip', 'eco_model (2).pkl', 'vectorizer.pkl', 'train.csv', 'test.csv', 'eco_model.pkl', 'vectorizer (2).pkl', 'eco_model (3).pkl', 'archive (1).zip', 'archive (1) (1).zip', 'vectorizer (1).pkl', 'drive', 'archive (1) (2).zip', 'archive (1) (4).zip', 'eco_model (4).pkl', 'eco_model (1).pkl', 'vectorizer (4).pkl', 'sample_data']


In [ ]:
amazon_df = pd.read_csv("train.csv")
flipkart_df = pd.read_csv("test.csv")

print("Amazon Dataset Shape:", amazon_df.shape)
print("Flipkart Dataset Shape:", flipkart_df.shape)

Amazon Dataset Shape: (15730, 16)
Flipkart Dataset Shape: (5244, 14)


In [ ]:
amazon_df.head()

,id,title,Rating,maincateg,platform,price1,actprice1,Offer %,norating1,noreviews1,star_5f,star_4f,star_3f,star_2f,star_1f,fulfilled1
0,16695,Fashionable & Comfortable Bellies For Women (...,3.9,Women,Flipkart,698,999,30.13%,38.0,7.0,17.0,9.0,6.0,3,3,0
1,5120,Combo Pack of 4 Casual Shoes Sneakers For Men ...,3.8,Men,Flipkart,999,1999,50.03%,531.0,69.0,264.0,92.0,73.0,29,73,1
2,18391,Cilia Mode Leo Sneakers For Women (White),4.4,Women,Flipkart,2749,4999,45.01%,17.0,4.0,11.0,3.0,2.0,1,0,1
3,495,Men Black Sports Sandal,4.2,Men,Flipkart,518,724,15.85%,46413.0,6229.0,1045.0,12416.0,5352.0,701,4595,1
4,16408,Men Green Sports Sandal,3.9,Men,Flipkart,1379,2299,40.02%,77.0,3.0,35.0,21.0,7.0,7,7,1


In [ ]:
flipkart_df.head()

,id,title,Rating,maincateg,platform,actprice1,norating1,noreviews1,star_5f,star_4f,star_3f,star_2f,star_1f,fulfilled1
0,2242,Casuals For Men (Blue),3.8,Men,Flipkart,999,27928,3543,14238.0,4295,3457,1962,3976.0,1
1,20532,Women Black Flats Sandal,3.9,Women,Flipkart,499,3015,404,1458.0,657,397,182,321.0,1
2,10648,Women Gold Wedges Sandal,3.9,Women,Flipkart,999,449,52,229.0,70,71,33,46.0,1
3,20677,Men's Height Increasing High Heel Formal Party...,3.9,Men,Flipkart,2999,290,40,141.0,51,49,17,32.0,1
4,12593,Loafers For Men (Tan),3.9,Men,Flipkart,999,2423,326,1265.0,414,293,143,308.0,0


In [30]:
# Amazon
amazon_df['text'] = amazon_df['title'].astype(str)

# Flipkart
flipkart_df['text'] = flipkart_df['title'].astype(str)

In [ ]:
amazon_df = amazon_df[['text']]
flipkart_df = flipkart_df[['text']]

In [31]:
eco_keywords = [
    "eco", "organic", "biodegradable", "recyclable",
    "bamboo", "natural", "sustainable", "plastic-free"
]

def label_text(text):
    text = text.lower()
    for word in eco_keywords:
        if word in text:
            return 1
    return 0

amazon_df['label'] = amazon_df['text'].apply(label_text)
flipkart_df['label'] = flipkart_df['text'].apply(label_text)

In [32]:
df = pd.concat([amazon_df, flipkart_df], ignore_index=True)

print("Total samples:", len(df))
df.head()

Total samples: 10


,title,maincateg,Offer %,price1,source,text,label,Rating
0,Eco-Friendly Bamboo Toothbrush,Personal Care,10.0,100.0,amazon,Eco-Friendly Bamboo Toothbrush,1,NaN
1,Organic Cotton T-Shirt,Apparel,20.0,500.0,amazon,Organic Cotton T-Shirt,1,NaN
2,Plastic Bottle 500ml,Kitchen,0.0,20.0,amazon,Plastic Bottle 500ml,0,NaN
3,Solar Power Bank,Electronics,15.0,1500.0,amazon,Solar Power Bank,0,NaN
4,Glass Mug,Kitchen,5.0,300.0,amazon,Glass Mug,0,NaN


In [36]:
df = pd.concat([amazon_df, flipkart_df], ignore_index=True)

print("Total samples:", len(df))
df.head()

Total samples: 10


,title,maincateg,Offer %,price1,source,text,label,Rating
0,Eco-Friendly Bamboo Toothbrush,Personal Care,10.0,100.0,amazon,Eco-Friendly Bamboo Toothbrush,1,NaN
1,Organic Cotton T-Shirt,Apparel,20.0,500.0,amazon,Organic Cotton T-Shirt,1,NaN
2,Plastic Bottle 500ml,Kitchen,0.0,20.0,amazon,Plastic Bottle 500ml,0,NaN
3,Solar Power Bank,Electronics,15.0,1500.0,amazon,Solar Power Bank,0,NaN
4,Glass Mug,Kitchen,5.0,300.0,amazon,Glass Mug,0,NaN


In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english')

X = vectorizer.fit_transform(df['text'])
y = df['label']

In [38]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [39]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)

LogisticRegression()

In [40]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.5
              precision    recall  f1-score   support

           0       0.50      1.00      0.67         1
           1       0.00      0.00      0.00         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [41]:
sample = ["eco friendly bamboo bottle"]

sample_vec = vectorizer.transform(sample)
prediction = model.predict(sample_vec)

print("Prediction:", prediction)

Prediction: [0]


In [42]:
import joblib

joblib.dump(model, "eco_model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")

['vectorizer.pkl']

In [43]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [44]:
import os

path = "/content/drive/MyDrive/eco_ai_project"
os.makedirs(path, exist_ok=True)

In [45]:
import joblib

joblib.dump(model, f"{path}/eco_model.pkl")
joblib.dump(vectorizer, f"{path}/vectorizer.pkl")

['/content/drive/MyDrive/eco_ai_project/vectorizer.pkl']

In [46]:
os.listdir(path)

['eco_model.pkl', 'vectorizer.pkl', 'rf_model.pkl', 'tfidf_vectorizer.pkl']

In [49]:
from google.colab import files

files.download(f"{path}/eco_model.pkl")
files.download(f"{path}/vectorizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [48]:
from google.colab import drive
from google.colab import files
import os
import joblib

# Define the path to your Google Drive folder
path = "/content/drive/MyDrive/eco_ai_project"

# Ensure Google Drive is mounted (force remount to refresh connection)
drive.mount('/content/drive', force_remount=True)

# Ensure the directory exists
os.makedirs(path, exist_ok=True)

# Re-save the model and vectorizer to the drive to ensure they are present
# This assumes 'rf_model' and 'tfidf_vectorizer' are still in the kernel's memory
try:
    joblib.dump(rf_model, f"{path}/rf_model.pkl")
    joblib.dump(tfidf_vectorizer, f"{path}/tfidf_vectorizer.pkl")
    print(f"Files re-saved successfully to {path}")
except NameError:
    print("Error: 'rf_model' or 'tfidf_vectorizer' not found in memory. Please ensure previous model training cells were run.")
except Exception as e:
    print(f"An error occurred while re-saving files: {e}")


# Attempt to download the files
if os.path.exists(f"{path}/rf_model.pkl") and os.path.exists(f"{path}/tfidf_vectorizer.pkl"):
    print(f"Attempting to download files from {path}...")
    files.download(f"{path}/rf_model.pkl")
    files.download(f"{path}/tfidf_vectorizer.pkl")
else:
    print("Files not found in Google Drive after attempting to re-save. Please check your Google Drive or rerun the model training and saving steps.")

Mounted at /content/drive
Files re-saved successfully to /content/drive/MyDrive/eco_ai_project
Attempting to download files from /content/drive/MyDrive/eco_ai_project...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>